# Observation fields and source definitions

`ObservationField` stores field-level observing metadata, while `SourceInfo`
describes one identified source within a field.

A field pointing is not necessarily the coordinate of a science target or
the WCS-derived center of an image.

In [5]:
import astropy.units as u
from astropy.coordinates import SkyCoord

from photozpy.sources import ObservationField, SourceInfo
from photozpy.telescope import SARA_RM

## Define an observation field

`nominal_pointing` is the planned telescope pointing. It does not need to
coincide with the coordinate of any source in the field.

`file_pattern` can later be used to associate image files with this field.

In [7]:
nominal_pointing = SkyCoord(
    ra=150.116321 * u.deg,
    dec=2.205830 * u.deg,
    frame="icrs",
)

science_field = ObservationField(
    field_name="SN2025abc_field",
    telescope=SARA_RM,
    nominal_pointing=nominal_pointing,
    file_pattern="SN2025abc",
)

In [9]:
print(f"Field name: {science_field.field_name}")
print(
    "Nominal pointing:",
    science_field.nominal_pointing.to_string(
        "hmsdms",
        sep=":",
        precision=2,
    ),
)
print(f"File pattern: {science_field.file_pattern}")
print(f"Telescope filters: {science_field.telescope.filters()}")

Field name: SN2025abc_field
Nominal pointing: 10:00:27.92 +02:12:20.99
File pattern: SN2025abc
Telescope filters: ("SDSS_u'", "SDSS_g'", "SDSS_r'", "SDSS_i'", "SDSS_z'")


## Define a science source

Each `SourceInfo` represents exactly one source. Its coordinate must therefore
be a scalar `SkyCoord`.

In [11]:
science_coordinate = SkyCoord(
    ra=150.120000 * u.deg,
    dec=2.210000 * u.deg,
    frame="icrs",
)

science_source = SourceInfo(
    source_name="SN2025abc",
    source_role="science",
    source_coordinate=science_coordinate,
    observation_field=science_field,
)

In [12]:
print(f"Source name: {science_source.source_name}")
print(f"Source role: {science_source.source_role}")
print(
    "Source coordinate:",
    science_source.source_coordinate.to_string(
        "hmsdms",
        sep=":",
        precision=2,
    ),
)
print(
    "Observation field:",
    science_source.observation_field.field_name,
)

Source name: SN2025abc
Source role: science
Source coordinate: 10:00:28.80 +02:12:36.00
Observation field: SN2025abc_field


## Field pointing and source coordinate are independent

The source coordinate is not required to coincide with the nominal telescope
pointing.

In [15]:
pointing_offset = science_source.source_coordinate.separation(
    science_field.nominal_pointing
)

print(
    "Offset from nominal pointing:",
    pointing_offset.to(u.arcsec),
)

Offset from nominal pointing: 20.0128 arcsec


## Science and standard sources can share the same field

A source role belongs to `SourceInfo`, not to `ObservationField`. Therefore,
one field can contain both science targets and standard stars.

In [17]:
standard_coordinate = SkyCoord(
    ra=150.130000 * u.deg,
    dec=2.220000 * u.deg,
    frame="icrs",
)

standard_source = SourceInfo(
    source_name="Gaia DR3 123456789",
    source_role="standard",
    source_coordinate=standard_coordinate,
    observation_field=science_field,
)

In [18]:
field_sources = (
    science_source,
    standard_source,
)

for source in field_sources:
    coordinate = source.source_coordinate.to_string(
        "hmsdms",
        sep=":",
        precision=2,
    )

    print(
        f"{source.source_name:24s} "
        f"{source.source_role:8s} "
        f"{coordinate}"
    )

SN2025abc                science  10:00:28.80 +02:12:36.00
Gaia DR3 123456789       standard 10:00:31.20 +02:13:12.00


In [20]:
# Both sources reference the same ObservationField object.
assert (
    science_source.observation_field
    is standard_source.observation_field
)

assert science_source.source_role == "science"
assert standard_source.source_role == "standard"

## Dedicated standard star fields

A dedicated standard field can initially exist without any `SourceInfo`.
There is no need to create an incomplete source with an empty name or
coordinate.

In [21]:
# %%
standard_field = ObservationField(
    field_name="PG_standard_field",
    telescope=SARA_RM,
    nominal_pointing=SkyCoord(
        ra=25.000000 * u.deg,
        dec=5.000000 * u.deg,
        frame="icrs",
    ),
    file_pattern="PG_standard",
)

print(f"Standard field: {standard_field.field_name}")

Standard field: PG_standard_field


A standard-star catalog can later search the image footprint. Once the
catalog provides a stable identifier and coordinate, the result can be
converted into a complete `SourceInfo`.

The following dictionary represents one mock catalog result.

In [23]:
catalog_result = {
    "source_name": "Gaia DR3 987654321",
    "ra_deg": 25.001200,
    "dec_deg": 5.002300,
}

discovered_standard = SourceInfo(
    source_name=catalog_result["source_name"],
    source_role="standard",
    source_coordinate=SkyCoord(
        ra=catalog_result["ra_deg"] * u.deg,
        dec=catalog_result["dec_deg"] * u.deg,
        frame="icrs",
    ),
    observation_field=standard_field,
)

print(f"Discovered standard: {discovered_standard.source_name}")
print(
    "Field:",
    discovered_standard.observation_field.field_name,
)

Discovered standard: Gaia DR3 987654321
Field: PG_standard_field


## Immutable metadata

`ObservationField` and `SourceInfo` are frozen dataclasses. Their attributes
cannot be reassigned after construction.

In [25]:
from dataclasses import FrozenInstanceError

try:
    science_source.source_name = "replacement"
except FrozenInstanceError as error:
    print(f"SourceInfo is immutable: {error}")

SourceInfo is immutable: cannot assign to field 'source_name'


## Scalar-coordinate requirement

A `SourceInfo` describes one source, so a vector `SkyCoord` is rejected.
Multiple sources should be represented by multiple `SourceInfo` objects.

In [27]:
multiple_coordinates = SkyCoord(
    ra=[150.12, 150.13] * u.deg,
    dec=[2.21, 2.22] * u.deg,
    frame="icrs",
)

try:
    SourceInfo(
        source_name="multiple_sources",
        source_role="science",
        source_coordinate=multiple_coordinates,
        observation_field=science_field,
    )
except ValueError as error:
    print(f"Invalid source coordinate: {error}")

Invalid source coordinate: 'source_coordinate' must contain exactly one coordinate.
